# Chatbot with Conversation Memory - Assignment

Welcome to your next assignment! This one has two parts, and both are required.

## Part 1 - Build the Backend

Complete the code in this notebook to get a working chatbot backend with **rolling memory**. The chatbot should remember the last 10 messages of the conversation. Older messages get dropped automatically as the conversation grows.

Your backend will expose three endpoints:

- `POST /chat` - send a message and get a response
- `GET /history` - retrieve the current conversation history
- `GET /clear` - reset the conversation

## Part 2 - Build a UI

Once your backend is running, build a user interface for it. It can be anything you like: a web page, a simple mobile layout, a desktop app, a CLI tool. The only requirement is that it connects to your `/chat` endpoint and shows the conversation.

**For your submission, include screenshots of your UI in action.** Show at least one multi-turn conversation that demonstrates the memory is working (i.e. the model refers back to something said earlier in the chat).

## What You Will Implement

- `add_to_memory(role, content)` - add a message to the history and enforce the 10-message limit
- `build_messages()` - assemble the full prompt list (system message + memory) for the model
- `POST /chat` route - the main endpoint that ties everything together

Everything else (model loading, Flask setup, other routes) is provided as scaffolding.


### Step 1 - Install Dependencies


In [ ]:
!pip install --upgrade --quiet pip
!pip install pyngrok flask flask-cors transformers --quiet
!pip install --quiet einops sentencepiece safetensors

import torch, transformers
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


### Step 2 - Import Libraries


In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
import torch, time


### Step 3 - Authenticate with Hugging Face

You need a Hugging Face token to download the model. You can paste it directly below, but think about whether that is a good idea before doing so. A safer option is to store it as a Kaggle secret and load it at runtime:

```python
from kaggle_secrets import UserSecretsClient
HUGGINGFACE_API_KEY = UserSecretsClient().get_secret("HF_TOKEN")
```

Either way works for this assignment, just be mindful of who can see your notebook.


In [ ]:
from kaggle_secrets import UserSecretsClient
HUGGINGFACE_API_KEY = UserSecretsClient().get_secret("HF_Token")

login(HUGGINGFACE_API_KEY)
print("Logged in to Hugging Face!")


### Step 4 - Authenticate with ngrok

Same story here. Pasting the token inline works, but storing it as a secret is cleaner:

```python
from kaggle_secrets import UserSecretsClient
NGROK_AUTH_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")
```


In [ ]:
NGROK_AUTH_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)


### Step 5 - Load the Model

This cell loads the tokenizer and model. It will take a few minutes the first time.


In [ ]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    low_cpu_mem_usage=True,
)

print(f"Model loaded: {model_name}")


### Step 6 - Build the Chatbot Backend

Here is your main task. Three things are left for you to implement, marked with `TODO`.

**`add_to_memory(role, content)`**
Adds a new message to `CHAT_MEMORY` and trims the list so it never holds more than `MEMORY_LIMIT` messages. The system message at index 0 must never be removed.

**`build_messages()`**
Returns the list of messages to pass to the model. It should be the system message followed by everything currently in `CHAT_MEMORY`.

**`POST /chat`**
The main endpoint. It should:
1. Read the user's message from the incoming JSON body (key: `message`).
2. Add it to memory.
3. Build the prompt and generate a reply.
4. Add the assistant's reply to memory.
5. Return a JSON response with the reply and the updated history.


In [ ]:
app = Flask(__name__)
CORS(app)

MEMORY_LIMIT = 10

SYSTEM_MESSAGE = {
    "role": "system",
    "content": "You are a helpful and friendly AI assistant. Keep your answers clear, concise and short.",
}

CHAT_MEMORY = []


def add_to_memory(role, content):
    """
    Append a new message dict to CHAT_MEMORY.
    Message dict: {"role": role, "content": content}
    """
   
    CHAT_MEMORY.append({"role": role, "content": content})
    
    """
    After appending, we use len(CHAT_MEMORY) to check no of messages stored
    if it exceeds MEMORY_LIMIT [10], we remove the
    oldest message of CHAT_MEMORY using pop(0).
    """
    while len(CHAT_MEMORY) > MEMORY_LIMIT:
        CHAT_MEMORY.pop(0)


def build_messages():
    """
    Return the full message list for the model by 
    combining the system instruction with the conversation history
    """
    return [SYSTEM_MESSAGE] + CHAT_MEMORY


def generate_reply(messages, max_new_tokens=258):
    """Generate a response from the model given a messages list."""
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


@app.route("/chat", methods=["POST"])
def chat():
    """
    The /chat endpoint handles a chat request.

    Following the steps:
      1. Parse the message from request.get_json().
         Return a 400 error if the message is missing or empty.
      2. Call add_to_memory("user", message).
      3. Call build_messages() and pass the result to generate_reply().
      4. Call add_to_memory("assistant", reply).
      5. Return jsonify({"reply": reply, "history": CHAT_MEMORY}).
    """
    # The endpoint receives a message from the user through a POST request
    data = request.get_json(silent=True) or {}
    message = data.get("message", "").strip()

    """
    Checking if the message is not empty. If empty, return an error message
    else stores the user's message, combines it with the existing conversation history 
    and sends everything to the model.
    """
    if not message:
        return jsonify({"error": "Please enter a message"}), 400

    add_to_memory("user", message)

    messages = build_messages()
    # knowing the time it took to reply
    start = time.time()
    reply = generate_reply(messages)

    print("Generation time:", round(time.time() - start, 2), "seconds")

    # store reply 
    add_to_memory("assistant", reply)

    # returns both response and updated conversation history as JSON
    return jsonify({
        "reply": reply,
        "history": CHAT_MEMORY
    })


@app.route("/history", methods=["GET"])
def history():
    """Return the current conversation history."""
    return jsonify({"history": CHAT_MEMORY})


@app.route("/clear", methods=["GET"])
def clear():
    """Reset the conversation."""
    CHAT_MEMORY.clear()
    return jsonify({"status": "cleared"})


### Step 7 - Launch the Server

Run this cell to start the server. Copy the public URL from the output and use it as the base URL in your UI (Part 2 of the assignment).

For example, to test your `/chat` endpoint from the terminal:

```bash
curl -X POST <your-url>/chat \
     -H "Content-Type: application/json" \
     -d '{"message": "Hello!"}'
```


In [ ]:
if __name__ == "__main__":
    ngrok.kill()
    public_url = ngrok.connect(5000)
    print(f"Public URL: {public_url.public_url}")
    print(f"Chat endpoint: {public_url.public_url}/chat")
    app.run(host="0.0.0.0", port=5000, debug=True, use_reloader=False)
